# **Gradient Boosting Classifier**

# In-Depth Guide: Scikit-Learn Stacking Hyperparameters

Scikit-Learn provides `StackingClassifier` and `StackingRegressor`. These classes share most of the same logic regarding how they combine models.


#### 1. `estimators` 
*   **Type:** `list` of `(str, estimator)` tuples.
*   **Description:** This defines your **Layer 0 (Base Models)**.
*   **Best Practice:** Choose a **diverse** set of models. Combining three different Random Forests is less effective than combining a Random Forest, an XGBoost, and a Logistic Regression.
*   **Example:** `estimators=[('rf', RandomForestClassifier()), ('xgb', XGBClassifier())]`


#### 2. `final_estimator`
*   **Type:** `estimator` object (Default: `LogisticRegression` for Classification, `RidgeCV` for Regression).
*   **Description:** The **Meta-Model (Layer 1)** that learns how to best combine the predictions of the base models.
*   **Advanced Tip:** While simple models are usually preferred to prevent overfitting, you can use a complex model here (like a Gradient Boosting machine) if you have a massive amount of base models.


#### 3. `cv`
*   **Type:** `int`, cross-validation generator, or iterable.
*   **Description:** **The most critical technical parameter.** 
*   It determines how the **Out-of-Fold (OOF)** predictions are generated.
*   **Logic:** The `Stacking` object uses this CV strategy to ensure that the `final_estimator` is trained on "unseen" predictions from the base models, preventing data leakage.
*   **Values:** 
    *   `None`: Uses default 5-fold cross-validation.
    *   `int`: Specifies the number of folds (e.g., `cv=10`).
    *   `prefit`: **Warning:** Only use this if the base models are *already fitted* on a separate validation set. It skips the OOF step.


#### 4. `stack_method` (Classifiers Only)
*   **Type:** `{'auto', 'predict_proba', 'decision_function', 'predict'}` (Default: `'auto'`).
*   **Description:** Determines what values are passed from the base models to the meta-model.
*   **In-Depth:**
    *   `'predict_proba'`: Passes the probabilities for each class (Highly recommended for better information density).
    *   `'decision_function'`: Passes the raw distance from the boundary.
    *   `'predict'`: Passes only the hard class labels (0, 1, 2...). Usually loses too much information.
    *   `'auto'`: Tries `predict_proba`, then `decision_function`, then `predict` in that order.


#### 5. `passthrough`
*   **Type:** `bool` (Default: `False`).
*   **Description:** If `True`, the meta-model will receive **both** the predictions from the base models **and** the original raw input features ($X$).
*   **When to use:** Use this if you suspect the raw features provide context that helps the meta-model decide which base model to trust (e.g., "In the North region, trust Model A; in the South, trust Model B").


#### 6. `n_jobs`
*   **Type:** `int` (Default: `None`).
*   **Description:** Number of CPU cores to use. 
*   **Note:** Because stacking involves training many models across many CV folds, setting `n_jobs=-1` can significantly speed up the process.


#### 7. `verbose`
*   **Type:** `int` (Default: `0`).
*   **Description:** Higher values give more log messages during the training of the multiple base models and the meta-model.


##### Summary Table for Quick Reference

| Parameter | Recommended Value | Impact on Performance |
| :--- | :--- | :--- |
| **`estimators`** | At least 3 diverse types | **High** (Foundation of the ensemble) |
| **`final_estimator`** | Simple (LR / Ridge) | **Medium** (Decides the blend) |
| **`cv`** | 5 or 10 | **High** (Prevents overfitting/leakage) |
| **`stack_method`**| `predict_proba` | **High** (Improves information flow) |
| **`passthrough`** | `False` | **Low** (Try `True` if accuracy stalls) |


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

import warnings
warnings.filterwarnings('ignore') 

In [2]:
df = pd.read_csv('heart.csv')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'heart.csv'

In [ ]:
df.nunique()

Age                50
Sex                 2
ChestPainType       4
RestingBP          67
Cholesterol       222
FastingBS           2
RestingECG          3
MaxHR             119
ExerciseAngina      2
Oldpeak            53
ST_Slope            3
HeartDisease        2
dtype: int64

In [ ]:
# using column transformer for categorical data
trf =  ColumnTransformer([
    ('t1',OneHotEncoder(drop='first', sparse_output=False), ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'])
], 
remainder='passthrough',
verbose_feature_names_out=False     # uncomment this line to see the difference
)


X = df.drop(columns=['HeartDisease'])
X = pd.DataFrame(trf.fit_transform(X), columns=trf.get_feature_names_out())

y = df['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)

print('FEATUES : ', X.shape[1])
X.head()

FEATUES :  15


,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak
0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,40.0,140.0,289.0,0.0,172.0,0.0
1,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,49.0,160.0,180.0,0.0,156.0,1.0
2,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,37.0,130.0,283.0,0.0,98.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,48.0,138.0,214.0,0.0,108.0,1.5
4,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,54.0,150.0,195.0,0.0,122.0,0.0


In [3]:
# making an object of different models

models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=10)),
    ('gbt', GradientBoostingClassifier())
]


# using stacking classifier
from sklearn.ensemble import StackingClassifier

clf = StackingClassifier(
    estimators=models,
    final_estimator=LogisticRegression(),       # this is our meta model
    cv=10,
    n_jobs=-1
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)


NameError: name 'X_train' is not defined

In [ ]:
print(f"ACCURACY_SCORE : {np.round(accuracy_score(y_test, y_pred),2)}")
print(f"CROSS VAL SCORE : {np.mean(np.round(cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy'),2))}")

# KEEP IN MIND -- doing cross val score will take time, as we are already using a heavy model and along with that if we use cross_val_score a huge cv, it will take a heavy computation



ACCURACY_SCORE : 0.86
CROSS VAL SCORE : 0.86


# **Gradient Boosting Regressor**

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=1000,
    n_features=6,
    noise=0.2,
    random_state=42
)

X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(6)])
y = pd.DataFrame(y, columns=['target'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)


X_train.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5
578,-1.724918,-0.562288,-1.012831,0.241962,-1.913280,0.314247
66,-0.305189,1.046912,-0.335244,-0.138512,-0.423890,-1.341065
39,-0.245743,-0.272724,-2.696887,-0.259591,-1.503143,-0.054295
302,-0.921860,-1.003957,0.207267,-0.083438,-1.449645,0.069344
974,-0.936506,-0.667780,0.292193,-0.031059,-0.909683,-0.187329


In [ ]:

models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('knn', KNeighborsRegressor(n_neighbors=10)),
    ('gbt', GradientBoostingRegressor())
]

from sklearn.ensemble import StackingRegressor

reg = StackingRegressor(
    estimators=models,
    final_estimator=LinearRegression(),
    cv=10,
    n_jobs=-1
)

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

print(f"R2_SCORE : {np.round(r2_score(y_test, y_pred), 2)}")
cv_score = cross_val_score(reg, X_train, y_train, cv=5, scoring='r2')
print(f"CROSS VAL SCORE : {np.mean(np.round(cv_score, 2))}")


R2_SCORE : 0.97
CROSS VAL SCORE : 0.974
